In [ ]:
print("jupyter notebook workin")

workin


#DATASET

In [ ]:
"""
Synthetic dynamic pricing dataset generator.

"""

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
N = 20000                # number of rows
SEED = 42
np.random.seed(SEED)

BASE_FARE = 40           # ₹ base fare
PER_KM_RATE = 18         # ₹ per km
ALPHA = 0.65             # weight: own surge-based price vs competitor-anchored price

# ---------------------------------------------------------
# 1. Categorical context features
# ---------------------------------------------------------
Season = np.random.choice(
    ["Normal", "Festival", "Monsoon", "Wedding", "Exam"],
    N,
    p=[0.55, 0.10, 0.20, 0.10, 0.05]
)

Day_Time = np.random.choice(
    ["Morning Peak", "Midday", "Evening Peak", "Night", "Late Night"],
    N,
    p=[0.22, 0.33, 0.25, 0.15, 0.05]
)

Discount = np.random.choice([0, 0.10, 0.20, 0.30], N)

CustomerRating = np.random.normal(4.3, 0.4, N).clip(1, 5).round(2)

Distance_km = np.random.uniform(1, 25, N).round(2)

# ---------------------------------------------------------
# 2. Demand
# ---------------------------------------------------------
base_demand = np.random.randint(80, 180, N)
demand_multiplier = np.ones(N)

# Time of day effect
demand_multiplier[Day_Time == "Morning Peak"] *= 1.5
demand_multiplier[Day_Time == "Midday"] *= 1.0
demand_multiplier[Day_Time == "Evening Peak"] *= 1.8
demand_multiplier[Day_Time == "Night"] *= 0.8
demand_multiplier[Day_Time == "Late Night"] *= 0.5

# Season effect
demand_multiplier[Season == "Festival"] *= 1.4
demand_multiplier[Season == "Wedding"] *= 1.3
demand_multiplier[Season == "Monsoon"] *= 1.2
demand_multiplier[Season == "Exam"] *= 0.85
demand_multiplier[Season == "Normal"] *= 1.0

Demand = (base_demand * demand_multiplier).astype(int)

# ---------------------------------------------------------
# 3. Stock / supply
# ---------------------------------------------------------
base_supply = np.random.randint(120, 250, N)
supply_multiplier = np.ones(N)

# Season effect
supply_multiplier[Season == "Festival"] *= 0.80
supply_multiplier[Season == "Wedding"] *= 0.85
supply_multiplier[Season == "Monsoon"] *= 0.75
supply_multiplier[Season == "Exam"] *= 1.05
supply_multiplier[Season == "Normal"] *= 1.00

# Time effect
supply_multiplier[Day_Time == "Morning Peak"] *= 0.90
supply_multiplier[Day_Time == "Midday"] *= 1.10
supply_multiplier[Day_Time == "Evening Peak"] *= 0.85
supply_multiplier[Day_Time == "Night"] *= 0.75
supply_multiplier[Day_Time == "Late Night"] *= 0.60

Stock = (base_supply * supply_multiplier).astype(int)

# ---------------------------------------------------------
# 4. Surge multiplier (computed BEFORE anything that depends on it)
# ---------------------------------------------------------
demand_stock_ratio = Demand / (Stock + 1)   # +1 avoids divide-by-zero

SurgeMultiplier = 1 + (
    (demand_stock_ratio - demand_stock_ratio.mean()) / demand_stock_ratio.std()
) * 0.5
SurgeMultiplier = SurgeMultiplier.clip(1.0, 3.0).round(3)

# ---------------------------------------------------------
# 5. Competitor price (depends on distance AND demand/stock, not just distance)
# ---------------------------------------------------------
CompetitorPrice = (
    40
    + Distance_km * PER_KM_RATE
    + (Demand / 100) * 15          # competitors also raise prices in high demand
    - (Stock / 100) * 10           # and lower them when supply is abundant
    + np.random.normal(0, 15, N)   # market noise, decouples from pure distance
).clip(50, 600).round(2)

# ---------------------------------------------------------
# 6. Historical sales (past demand for this route/time, noisy proxy)
# ---------------------------------------------------------
HistoricalSales = (base_demand * np.random.uniform(0.8, 1.2, N)).astype(int)

# ---------------------------------------------------------
# 7. Final price: own surge-based price blended with competitor anchoring
# ---------------------------------------------------------
distance_fare = BASE_FARE + Distance_km * PER_KM_RATE

# surge premium anchored to an "average" trip so it doesn't scale with THIS
# trip's distance (keeps surge as its own independent signal)
surge_premium = (SurgeMultiplier - 1) * (BASE_FARE + 12 * PER_KM_RATE)

own_price = (distance_fare + surge_premium) * (1 - Discount)

Price = (
    ALPHA * own_price
    + (1 - ALPHA) * CompetitorPrice
    + np.random.normal(0, 8, N)     # irreducible noise
).clip(50, 900).round(2)

# ---------------------------------------------------------
# 8. Assemble dataframe
# ---------------------------------------------------------
df = pd.DataFrame({
    "Demand": Demand,
    "Stock": Stock,
    "CompetitorPrice": CompetitorPrice,
    "Season": Season,
    "DayTime": Day_Time,
    "CustomerRating": CustomerRating,
    "Discount": Discount,
    "HistoricalSales": HistoricalSales,
    "Distance_km": Distance_km,
    "SurgeMultiplier": SurgeMultiplier,
    "Price": Price,
})

# ---------------------------------------------------------
# 9. Quick sanity check + save
# ---------------------------------------------------------
if __name__ == "__main__":
    print(df.head())
    print("\nShape:", df.shape)
    print("\nCorrelation with Price:")
    print(df.corr(numeric_only=True)["Price"].sort_values(ascending=False))

    df.to_csv("dynamic_pricing_data.csv", index=False)
    print("\nSaved to dynamic_pricing_data.csv")

   Demand  Stock  CompetitorPrice    Season       DayTime  CustomerRating  \
0     187    135           286.31    Normal  Evening Peak            4.29   
1     182    216           100.62      Exam  Morning Peak            4.78   
2     100    100            96.09   Monsoon        Midday            4.55   
3     357    146           325.07  Festival  Evening Peak            3.98   
4     164    215           326.14    Normal        Midday            5.00   

   Discount  HistoricalSales  Distance_km  SurgeMultiplier   Price  
0       0.3               92        12.56            1.079  230.50  
1       0.0              169         3.42            1.000   85.48  
2       0.0               84         2.65            1.000   99.91  
3       0.3              160        13.26            1.814  331.76  
4       0.2              190        15.87            1.000  286.01  

Shape: (20000, 11)

Correlation with Price:
Price              1.000000
CompetitorPrice    0.909362
Distance_km        0.8